# Annexe B — Cahier de code, Chapitre 17
## Descripteurs locaux et appariement

Ce notebook accompagne le chapitre 17 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Une image et sa version transformée (rotation), pour apparier des points
import numpy as np
from skimage import data, color, transform

img = color.rgb2gray(data.astronaut())
img2 = transform.rotate(img, 30, resize=False)

## 17.1 — Le problème de l'appariement

In [ ]:
# comparer des imagettes brutes échoue dès qu'il y a rotation/échelle
patch1 = img[100:120, 100:120]
patch2 = img2[100:120, 100:120]
print("différence brute :", np.abs(patch1 - patch2).mean())

## 17.2 — Échelle caractéristique

In [ ]:
# détecter à quelle taille un point est saillant (blobs DoG)
from skimage.feature import blob_dog
blobs = blob_dog(img, max_sigma=30, threshold=0.1)   # (y, x, sigma)

## 17.3 — HOG

In [ ]:
# histogramme des orientations de gradient (descripteur de fenêtre)
from skimage.feature import hog
vecteur, visu = hog(img, orientations=9, pixels_per_cell=(16, 16),
                    cells_per_block=(2, 2), visualize=True)

## 17.4 — SIFT

In [ ]:
# détection + description invariantes échelle/rotation
from skimage.feature import SIFT
sift = SIFT()
sift.detect_and_extract(img)
kp, desc = sift.keypoints, sift.descriptors

## 17.5 — ORB et BRIEF

In [ ]:
# descripteurs binaires, rapides
from skimage.feature import ORB
orb = ORB(n_keypoints=200)
orb.detect_and_extract(img)
kp, desc = orb.keypoints, orb.descriptors

## 17.6 — Le ratio test de Lowe

In [ ]:
# garder un appariement seulement si le meilleur est nettement devant le 2e
from skimage.feature import SIFT, match_descriptors
s1, s2 = SIFT(), SIFT()
s1.detect_and_extract(img); s2.detect_and_extract(img2)
matches = match_descriptors(s1.descriptors, s2.descriptors,
                            max_ratio=0.7, cross_check=True)
print(len(matches), "bons appariements")

## 17.7 — RANSAC et homographie

In [ ]:
# imposer la cohérence géométrique entre les points appariés
from skimage.measure import ransac
from skimage.transform import ProjectiveTransform
src = s1.keypoints[matches[:, 0]][:, ::-1]   # (x, y)
dst = s2.keypoints[matches[:, 1]][:, ::-1]
modele, inliers = ransac((src, dst), ProjectiveTransform,
                         min_samples=4, residual_threshold=3, max_trials=200)
print(inliers.sum(), "appariements cohérents")

## 17.8 — L'état de l'art (deep learning)

In [ ]:
# détecteurs/descripteurs appris : SuperPoint, SuperGlue, LoFtR...
# disponibles via la librairie kornia (pip install kornia)
print("voir kornia.feature (SuperPoint, LoFTR) pour l'appariement appris")